
# <center>Designing the MLMC runner</center>

This notebook develops the initial design of the MLMC runner.

The runner is the orchestration layer connecting a user-defined multilevel model to the existing correction, solver, and statistics components. It schedules correction samples, creates reproducible random-number streams, invokes the correction calculation, accumulates correction-level statistics, and returns the final MLMC result.

Future parallelization/multithreading will be taken into account when designing the runner

## <u> Current package architecture </u>

The package already separates four important responsibilities.

| Component | Responsibility | Explicitly does not do |
|---|---|---|
| `linear_solver.py` | Solve one supplied linear system and return diagnostics | Know about MLMC levels, randomness, coupling, or estimators |
| `mlmc_model.py` | Define how a user model samples randomness, couples inputs, builds systems, and evaluates a quantity of interest | Schedule samples, solve corrections, or accumulate statistics |
| `mlmc_correction.py` | Compute one coupled correction sample | Decide how many samples to run or combine multiple levels |
| `mlmc_statistics.py` | Accumulate scalar correction and cost statistics online | Retain solution arrays or schedule work |

The missing layer is the runner. It repeatedly requests correction samples, gives each sample a reproducible identity, updates statistics, and constructs the final estimator.

## <u> Runner at a glance </u>

The runner coordinates sample scheduling, deterministic random-number generation, correction evaluation, statistics accumulation, and result construction.

It does not construct model inputs, assemble linear systems, calculate quantities of interest, implement linear solvers, or retain solution vectors. Those responsibilities remain with the model, correction evaluator, and solver layers.


<pre>
User supplies
┌──────────────────────────────────────────────┐
│ MultilevelModel                              │
│ SystemSolver                                 │
│ Base seed                                    │
│ Fixed sample counts or stopping criterion    │
└──────────────────────┬───────────────────────┘
                       │
                       ▼
                  MLMCRunner
┌──────────────────────────────────────────────┐
│ Validate the run                             │
│ Assign (level, sample_index) tasks           │
│ Construct deterministic RNGs                 │
│ Request coupled SampleCorrection objects     │
│ Update correction-level statistics           │
│ Assemble the final result                    │
└──────────────────────┬───────────────────────┘
                       │
                       ▼
                  MLMCResult
┌──────────────────────────────────────────────┐
│ Correction-level statistics                  │
│ MLMC estimate                                │
│ Estimated sampling variance and error        │
│ Per-level and total measured cost            │
└──────────────────────────────────────────────┘
</pre>

<style>
  .jp-RenderedMarkdown h3, h3 {
    color: #2e6f40 !important;
  }
</style>

<style>
  .jp-RenderedMarkdown h4, h4 {
    color:rgb(70, 90, 242) !important;
  }
</style>


## <u> General user-facing workflow </u>
<i> current + proposed runner</i>

</br>

### 1. Define a multilevel model

1. User defines a `MultilevelModel` that follows the existing protocol and provides:
    - number_of_levels
    - sample_randomness()
    - couple_inputs()
    - build_linear_system()
    - quantity_of_interest()
The model defines the multilevel problem and how adjacent levels are coupled. The runner will have no control over how coupling is done, how randomness is drawn, etc. It will just include some baseline checks to ensure consistency and size matching

### 2. Choose a linear-system solver
The user chooses and configures a solver that follows the existing `SystemSolver` interface:

```text
LinearSystem → LinearSolveResult
```

The initial runner can accept a solver callable directly. After the runner is stable, The available solvers will be expanded and the solver-configuration interface will be cleaned up and simplified.

### 3. Create an MLMC runner
The user creates an `MLMCRunner` and supplies:

```text
model       → defines the multilevel problem
solver      → solves each generated linear system
base_seed   → controls reproducible random sampling
```

### 4. Supply sample counts or a stopping criterion

The first runner implementation will execute fixed, user-specified sample counts at consecutive correction levels from level 0 through a selected finest level. For example:

```text
runner.run_fixed([1000, 200, 50], finest_level=2)
```

This requests:

- 1,000 samples of $Y_0=Q_0$
- 200 samples of $Y_1=Q_1-Q_0$
- 50 samples of $Y_2=Q_2-Q_1$

These are sample counts, not linear-solver iteration counts. Because the correction levels form the complete telescoping sum from 0 through 2, this run estimates $\mathbb{E}[Q_2]$.

The `finest_level` argument is optional. When it is `None`, the runner uses the model's finest available level. The number of supplied counts must always equal `finest_level + 1`, so every correction in the selected telescoping sum receives an initial sample count.

A later runner method will add the ability for adaptive sampling, running until a specified sampling-error tolerance or safety limit is reached.

### 5. Validate the requested run

Before performing any solves, the runner will go through several validations, such as:

- The selected finest level is available in the model.
- The count sequence has exactly one entry for every level from 0 through the selected finest level.
- Every initial sample count is a valid integer meeting the required minimum.

### 6. Run initialization

When starting a new run, the runner creates one `LevelStatistics` object for every correction level from 0 through the selected finest level. When adding samples to an existing run, it reuses the existing statistics.
The next sample index at a correction level is its current sample count. If level $\ell$ already contains $N_\ell$ samples, the next additional sample receives $ \text{sample\_index } = N_\ell$.

The public `run_fixed()` method starts a new run. The public `add_samples()` method extends the active run without resetting its statistics or reusing earlier sample indices. Calling `run_fixed()` again on an active runner should be rejected so existing results are not silently discarded.

### 7. Compute and accumulate correction samples

Every correction sample is identified by a `(level, sample_index)` pair:

```plain text
base seed + level + sample index
                │
                ▼
    deterministic sample RNG
                │
                ▼
    compute_sample_correction(
        model,
        fine_level=level,
        rng,
        solver,
    )
                │
                ▼
         SampleCorrection
```

The combination of the base seed, correction level, and sample index gives every correction sample a stable random identity.

#### Inside compute_sample_correction():
The runner will repeatedly call compute_sample_correction (in mlmc_correction.py) For correction level $\ell$, the correction calculation:

1. Uses the supplied RNG to draw one random realization.
2. Constructs coupled model inputs for levels $\ell$ and $\ell-1$ when $\ell>0$.
3. Builds the fine and optional coarse linear systems.
4. Uses the selected solver to solve those systems.
5. Evaluates $Q_\ell$ and, when required, $Q_{\ell-1}$.
6. Forms

   $$
   Y_0=Q_0
   $$

   or

   $$
   Y_\ell=Q_\ell-Q_{\ell-1}, \qquad \ell>0.
   $$

7. Returns a `SampleCorrection` containing the correction data, solver diagnostics, and elapsed time.

The runner does not repeat this fine/coarse logic. It delegates the complete one-sample operation to `compute_sample_correction()`.

#### Accumulating the sample

The runner passes the returned `SampleCorrection` to the matching `LevelStatistics`. This updates:

- The correction count
- The mean correction
- The correction sample variance
- The variance of the correction mean
- The mean correction cost
- The total correction cost

After the update, the runner does not retain the `SampleCorrection` or its fine and coarse solution arrays.


### 8. Return an MLMC result

After all requested samples have been completed, the runner returns an immutable `MLMCResult` snapshot. If more samples are later added, the runner returns a new snapshot; previously returned results must not change.

The result contains the correction-level statistics and provides the MLMC estimate

$$
\widehat Q_{\mathrm{ML}}
=
\sum_{\ell=0}^{L}\overline{Y}_\ell,
$$

the estimated sampling variance

$$
\widehat{\operatorname{Var}}
\left[\widehat Q_{\mathrm{ML}}\right]
=
\sum_{\ell=0}^{L}\frac{s_\ell^2}{N_\ell},
$$

the estimated standard error, and the total measured cost.

## Runner at a glance

<pre>
User supplies
┌──────────────────────────────────────────────┐
│ MultilevelModel                              │
│ SystemSolver                                 │
│ Base seed                                    │
│ Fixed sample counts or stopping criterion    │
└──────────────────────┬───────────────────────┘
                       │
                       ▼
                  MLMCRunner
┌──────────────────────────────────────────────┐
│ Validate the run                             │
│ Assign (level, sample_index) tasks           │
│ Construct deterministic RNGs                 │
│ Request coupled SampleCorrection objects     │
│ Update correction-level statistics           │
│ Assemble the final result                    │
└──────────────────────┬───────────────────────┘
                       │
                       ▼
                  MLMCResult
┌──────────────────────────────────────────────┐
│ Correction-level statistics                  │
│ MLMC estimate                                │
│ Estimated sampling variance and error        │
│ Per-level and total measured cost            │
└──────────────────────────────────────────────┘
</pre>

## <u>Proposed public class: `MLMCRunner`</u>

One public runner class should support multiple execution policies over time. The initial method will run fixed sample counts. A later method will add samples until a statistical tolerance is reached. Both modes should reuse the same sample-generation and accumulation machinery.

### Constructor state

| State | Meaning |
|---|---|
| `model` | User-supplied object satisfying `MultilevelModel` |
| `base_seed` | Root value used to identify deterministic sample streams |
| `solver` | Callable or configuration used to solve each generated `LinearSystem` |

The model already reports its available number of levels, so the runner should not duplicate that value as independent mutable state.

### Public methods

| Method | Responsibility |
|---|---|
| `run_fixed(samples_per_level, finest_level=None)` | Start a new run through an explicitly selected finest level, or through the model's finest level when omitted; reject the call if a run is already active |
| `add_samples(additional_samples_per_level)` | Add nonnegative sample counts to the currently active correction levels |
| `result` | Return a snapshot of the current accumulated result |
| `run_to_tolerance(...)` | Add samples adaptively until the target is met |

`run_fixed()` creates the initial correction-level statistics for levels 0 through the selected finest level. Its count sequence must contain exactly `finest_level + 1` entries. If `finest_level` is `None`, the runner selects `model.number_of_levels - 1`, so the count sequence must cover the entire model hierarchy.

`add_samples()` requires an active run, accepts one nonnegative count per active level, and continues from each level's current sample count. Zero means that an active level receives no new samples in that call. At least one count must be positive, and this method does not change the active finest level. A user who wants an independent run or a different finest level should create a new runner.

Keeping these operations on one runner gives users one coherent entry point without mixing the internal policies. Plotting and reporting conveniences can be designed later around the returned result rather than being required runner responsibilities.



## 5. Internal runner operations

The runner will need a small number of internal operations. Their exact names can change during implementation, but their responsibilities should remain separate.

| Internal operation | Responsibility |
|---|---|
| Resolve the finest level | Use the explicit available level, or the model's finest level when `None` |
| Validate initial sample counts | Require exactly one valid initial count for every level from 0 through the selected finest level |
| Validate additional sample counts | Require one count per active level, allow zero for unchanged levels, and require at least one positive count |
| Construct a sample RNG | Derive one generator from the base seed, correction level, and sample index |
| Compute one identified sample | Call `compute_sample_correction()` for one `(level, sample_index)` task |
| Accumulate one sample | Update only the matching `LevelStatistics` |
| Assemble a result | Freeze the final per-level statistics and expose aggregate quantities |
| Initialize run statistics | Create statistics only when a correction level is first activated |
| Determine next sample index | Use the existing correction sample count |
| Run an additional batch | Add samples to existing statistics without restarting their indices |
| Create a result snapshot | Return current results without exposing mutable accumulator state |

The one-sample operation is the unit that can later be sent to a parallel worker. Designing it explicitly now avoids changing the mathematical workflow when parallel scheduling is introduced.

## 6. Fixed-sample execution policy

A fixed-sample run receives one requested sample count per correction level from 0 through `finest_level`. For example, `run_fixed([100, 20, 5], finest_level=2)` estimates $Y_0$, $Y_1$, and $Y_2$ using those respective numbers of independent correction samples, and their telescoping sum estimates $\mathbb{E}[Q_2]$.

If `finest_level=None`, the runner uses the finest level supplied by the model. For a model with levels 0 through 4, `run_fixed([100, 20, 5])` is therefore invalid unless `finest_level=2` is supplied; a default full-hierarchy run requires five initial counts.

The logical execution order is:

1. Resolve and validate the selected finest level, then validate the sequence of sample counts.
2. Create an empty `LevelStatistics` object only when starting a new correction level. Reuse existing statistics when adding samples.
3. Visit correction levels in increasing order.
4. Visit sample indices at each level in increasing order.
5. Construct the deterministic RNG identified by that pair.
6. Compute one `SampleCorrection`.
7. Update the matching correction-level statistics.
8. Return an immutable result after all requested work succeeds.

### Adding fixed samples to an existing run

An additional-sample request reuses the active `LevelStatistics` objects. If level 1 already contains five samples with indices 0 through 4, requesting three additional samples assigns indices 5, 6, and 7. The additional batch therefore begins at `level_statistics.sample_count`. A zero count leaves an active level unchanged, so a request such as `[10, 0, 5]` adds samples only at levels 0 and 2. An all-zero request is rejected because it schedules no work.

`run_fixed()` and `add_samples()` both use the same internal sample-batch operation. The difference is whether the statistics are newly initialized or already contain samples.

The initial implementation is serial. The explicit `(level, sample index)` task identity is nevertheless chosen now so that later parallel execution changes only scheduling, not random values or statistical meaning.

## 7. Deterministic random-stream design

Every correction sample is identified by three integers:

- the runner's base seed;
- the correction level;
- the sample index within that level.

A `SeedSequence` derived from those values will create the sample's generator. This gives the following invariants:

- Repeating a run with the same base seed and sample counts reproduces the same correction values.
- Running $N_\ell$ samples and then adding $M_\ell$ samples produces the same correction sequence as requesting $N_\ell+M_\ell$ samples in one run.
- Increasing the number of samples at one level does not change samples already assigned to another level.
- Parallel task completion order cannot change which random realization belongs to a task.
- Fine and coarse evaluations inside one correction still share one random realization because `compute_sample_correction()` calls the model's randomness operation only once.

Elapsed times are not expected to be reproducible. Reproducibility tests should compare correction values and derived estimates, not measured costs.

## 8. Proposed result classes

The runner owns mutable `LevelStatistics` accumulators while a run is active. Those accumulators must remain available so `add_samples()` and the future adaptive policy can continue updating them.

A returned result should instead be an immutable snapshot. Otherwise, adding samples after returning a result could unexpectedly change that earlier result because `LevelStatistics` is mutable.

<pre>
MLMCRunner
    └── owns mutable LevelStatistics accumulators
                    │
                    │ snapshot
                    ▼
              immutable LevelResult objects
                    │
                    ▼
                 MLMCResult
</pre>

### `LevelResult`

One `LevelResult` stores scalar snapshot values for a correction level:

| Field | Meaning |
|---|---|
| `level` | Correction-level index |
| `sample_count` | Number of completed correction samples |
| `mean_correction` | Sample mean of $Y_\ell$ |
| `sample_variance` | Unbiased sample variance of $Y_\ell$ |
| `variance_of_mean` | Estimated variance of the correction mean |
| `mean_sample_cost` | Mean elapsed time per correction sample |
| `total_sample_cost` | Total elapsed time at the correction level |

### `MLMCResult`

`MLMCResult` stores the selected `finest_level` and an ordered tuple of immutable `LevelResult` snapshots for correction levels 0 through that level. Aggregate quantities are calculated from those snapshots rather than stored redundantly. The recorded finest level makes it explicit that the telescoping estimator targets $Q_L$.

### Derived aggregate properties

The MLMC estimate is

$$
\widehat Q_{\mathrm{ML}} = \sum_{\ell=0}^{L}\overline{Y}_\ell.
$$

The estimated sampling variance is

$$
\widehat{\operatorname{Var}}\!\left[\widehat Q_{\mathrm{ML}}\right]
= \sum_{\ell=0}^{L}\frac{s_\ell^2}{N_\ell}.
$$

The estimated standard error is the square root of this variance. Total measured cost is the sum of the correction-level costs.

If any level has fewer than two samples, its unbiased sample variance is undefined. The design must explicitly decide whether the result reports `NaN` for estimator variance or whether the runner requires at least two samples per requested level.

Creating a new snapshot after `add_samples()` must not mutate or otherwise change any previously returned `MLMCResult`.

## 9. Validation and failure behavior

Validation should occur at the layer that owns the relevant decision.

| Condition | Owning layer | Expected behavior |
|---|---|---|
| Explicit `finest_level` is not an available model level | Runner | Reject before sampling |
| `finest_level=None` | Runner | Select `model.number_of_levels - 1` |
| Initial count sequence length is not `finest_level + 1` | Runner | Reject before sampling |
| Noninteger initial count or count below the required minimum | Runner | Reject before sampling |
| Negative additional sample count | Runner | Reject before sampling |
| All additional sample counts are zero | Runner | Reject because no work was requested |
| Additional counts do not match the active levels | Runner | Reject before modifying statistics |
| `add_samples()` called before a run is started | Runner | Reject because no active statistics exist |
| `run_fixed()` called when a run is already active | Runner | Reject and direct the user to `add_samples()` or a new runner |
| Invalid correction level or fine/coarse structure | Correction evaluator | Reject while computing the sample |
| Failed fine or coarse solve | Correction evaluator | Raise with the failed level and solver message |
| Nonfinite correction or cost | Statistics layer | Reject without creating inconsistent counts |

A failed run should not return a result that looks complete. Persistence and restart behavior are later concerns and should not complicate the first serial implementation.

## 10. Solver configuration boundary

The runner owns which solver callable is supplied to `compute_sample_correction()`. The low-level solver continues to own tolerances, preconditioners, and iteration limits for an individual linear system.

This distinction prevents two unrelated meanings of iteration from becoming mixed:

- Linear-solver iterations are steps taken by CG or another iterative method for one system.
- MLMC sampling rounds are batches of additional correction samples requested by a future adaptive policy.

The initial runner can accept a solver callable. More elaborate solver-configuration objects should be added only if repeated use demonstrates that they are necessary.

## 11. Future tolerance-based execution

A later `run_to_tolerance()` method should reuse the same deterministic sample identities, one-sample correction operation, and `LevelStatistics`. It will add policy for deciding which level receives the next samples.

The first adaptive target should be stated precisely as a sampling-error target, for example

$$
\sum_{\ell=0}^{L}\frac{s_\ell^2}{N_\ell}\leq \varepsilon_{\mathrm{sampling}}^2.
$$

This does not by itself control the discretization bias between $Q_L$ and the limiting quantity $Q$. A complete RMSE target requires both sampling-error control and bias estimation, potentially followed by adding finer levels.

A practical later sequence is:

1. Run a small pilot sample count at every active correction level.
2. Estimate correction variances and mean costs.
3. Allocate additional samples using both variance and cost.
4. Continue sample indices rather than restarting streams.
5. Stop at the sampling target or a clearly defined safety budget.
6. Add bias estimation and automatic level extension as a separate milestone.

## 12. Testing plan before implementation

The synthetic linear model gives known coupling and inexpensive systems, making it suitable for runner tests.

### Fixed-run correctness

- `finest_level=None` activates every level provided by the model.
- An explicit lower `finest_level` activates exactly levels 0 through that level.
- `MLMCResult.finest_level` records the selected target level.
- Requested sample counts appear in the matching correction-level statistics.
- The reported estimate equals the sum of the correction means.
- The estimator variance equals the sum of the level mean variances.
- Total cost equals the sum of the correction-level total costs.

### Reproducibility

- The same base seed gives the same correction means and estimate.
- A chosen `(level, sample index)` produces the same random realization whenever it is requested.
- Increasing one level's sample count does not alter another level's correction samples.
- Running $N_\ell$ samples and then adding $M_\ell$ samples produces the same correction values as one run of $N_\ell+M_\ell$ samples.
- Tests do not require elapsed times to match.

### Continuing an existing run

- Additional samples begin at the current `sample_count` and never reuse an earlier sample index.
- Zero is accepted for an active level that should receive no additional samples, but at least one level must request a positive count.
- Existing `LevelStatistics` counts, means, variances, and costs are updated rather than replaced.
- `add_samples()` is rejected before a run has been initialized.
- A second `run_fixed()` call on an active runner is rejected rather than silently resetting statistics.
- A previously returned `MLMCResult` snapshot does not change after additional samples are accumulated.

### Validation and propagation

- An unavailable finest level, a count-length mismatch, or an invalid initial count is rejected.
- Initial counts cannot omit a correction within levels 0 through the selected finest level.
- A correction failure propagates rather than being silently omitted.
- The runner does not retain `SampleCorrection` objects or solution arrays after accumulation.

## 13. Design decisions

Some interface decisions have now been settled, while a few implementation choices remain open.

### Settled execution-policy decisions

- `run_fixed(samples_per_level, finest_level=None)` uses the model's finest available level when `finest_level` is omitted. This is estimating `finest_level`, not necessarily the finest level of the model 
- An explicit `finest_level=L` deliberately targets $Q_L$ and requires exactly $L+1$ initial counts for $Y_0,\ldots,Y_L$.
- Initial counts must meet the minimum at every selected level; gaps would break the complete telescoping estimator.
- `add_samples()` requires one count per already-active level, allows zero at any of those levels, and requires at least one positive count overall.
- `add_samples()` does not change the active finest level.

For a model with levels 0 through 4:

- `run_fixed([100, 20, 5], finest_level=2)` is valid and estimates $\mathbb{E}[Q_2]$.
- `run_fixed([100, 20, 5])` is invalid because the default target is $Q_4$ and five counts are required.
- `run_fixed([100, 20, 5, 2, 2])` is a valid default full-hierarchy run estimating $\mathbb{E}[Q_4]$.
- After that full run, `add_samples([0, 0, 100, 20, 5])` is valid for estimating $\mathbb{E}[Q_4]$ and leaves levels 0 and 1 unchanged. It is invalid when `finest_level < 4`. 

The initial run will require at least two samples per selected level so every sample variance is defined. A later `add_samples()` call may add only one sample at a level. The sample-RNG constructor will be a module-level internal helper called by the runner.

### Remaining open decisions

1. Should the runner accept only a solver callable for the MVP, with configured solver factories or objects deferred?
2. If a batch fails after some samples succeed, should those successfully accumulated samples remain in the active run?

Recording these decisions here keeps the package implementation intentional and makes the reasoning easy to transfer into formal project notes.

## 14. Proposed implementation sequence

Once the open decisions are resolved, implementation should proceed in narrow, testable stages:

1. Define and test deterministic sample identity and RNG construction.
2. Define immutable `LevelResult` and `MLMCResult` snapshots and their derived aggregate properties.
3. Define `MLMCRunner` with constructor validation and internal mutable correction-level statistics.
4. Implement `run_fixed()` and the reusable sample-batch operation.
5. Implement `add_samples()` using the same sample-batch operation and continuing sample indices.
6. Test the serial runner end to end with the synthetic model.
7. Add the runner to the teaching notebook and package documentation.
8. Design and implement sampling-tolerance allocation.
9. Add bias estimation and finer-level extension.
10. Add parallel scheduling while preserving identical sample identities and deterministic accumulation order.

The fixed-sample method is not a separate final architecture. It is the first execution policy and the foundation used by the later adaptive policy.

## 15. Initial runner framework stubs

The following cells introduce only the proposed names, relationships, type signatures, and short responsibilities. Every operation is deliberately unimplemented and uses an ellipsis. The stubs will be filled in one component at a time after their interfaces are reviewed.

In [ ]:
from collections.abc import Sequence
from dataclasses import dataclass
from typing import Generic, TypeVar

import numpy as np

from mlmc_linear_systems.linear_solver import solve_linear_system
from mlmc_linear_systems.mlmc_correction import (
    SampleCorrection,
    SystemSolver,
)
from mlmc_linear_systems.mlmc_model import MultilevelModel
from mlmc_linear_systems.mlmc_statistics import LevelStatistics


RandomnessT = TypeVar("RandomnessT")
ModelInputT = TypeVar("ModelInputT")

In [ ]:
def _make_sample_rng(
    base_seed: int,
    fine_level: int,
    sample_index: int,
) -> np.random.Generator:
    """Create the deterministic RNG for one identified correction sample."""
    ...

In [ ]:
@dataclass(frozen=True)
class LevelResult:
    """Immutable scalar snapshot of one correction level."""

    level: int
    sample_count: int
    mean_correction: float
    sample_variance: float
    variance_of_mean: float
    mean_sample_cost: float
    total_sample_cost: float


@dataclass(frozen=True)
class MLMCResult:
    """Immutable correction-level snapshots and derived MLMC estimates."""

    finest_level: int
    level_results: tuple[LevelResult, ...]

    @property
    def estimate(self) -> float:
        """Return the sum of the correction-level sample means."""
        ...

    @property
    def estimator_variance(self) -> float:
        """Return the estimated sampling variance of the MLMC mean."""
        ...

    @property
    def standard_error(self) -> float:
        """Return the estimated standard error of the MLMC mean."""
        ...

    @property
    def total_cost(self) -> float:
        """Return the total measured cost across all correction levels."""
        ...

In [ ]:
class MLMCRunner(Generic[RandomnessT, ModelInputT]):
    """Coordinate reproducible correction sampling and MLMC statistics."""

    _level_statistics: list[LevelStatistics]
    _finest_level: int | None

    def __init__(
        self,
        model: MultilevelModel[RandomnessT, ModelInputT],
        *,
        base_seed: int,
        solver: SystemSolver = solve_linear_system,
    ) -> None:
        """Store the model, base seed, and configured system solver."""
        ...

    def _validate_initial_sample_counts(
        self,
        samples_per_level: Sequence[int],
        finest_level: int,
    ) -> tuple[int, ...]:
        """Validate and normalize positive counts for a new run."""
        ...

    def _validate_additional_sample_counts(
        self,
        additional_samples_per_level: Sequence[int],
    ) -> tuple[int, ...]:
        """Validate nonnegative counts for the active correction levels."""
        ...

    def _initialize_run(
        self,
        finest_level: int,
    ) -> None:
        """Create empty statistics when starting a new run."""
        ...

    def _compute_sample(
        self,
        fine_level: int,
        sample_index: int,
    ) -> SampleCorrection:
        """Compute one correction identified by level and sample index."""
        ...

    def _run_sample_batch(
        self,
        level_statistics: LevelStatistics,
        *,
        start_index: int,
        sample_count: int,
    ) -> None:
        """Add a consecutive batch of samples to one correction level."""
        ...

    def _create_result(self) -> MLMCResult:
        """Create an immutable snapshot of the active run."""
        ...

    def run_fixed(
        self,
        samples_per_level: Sequence[int],
        *,
        finest_level: int | None = None,
    ) -> MLMCResult:
        """Start a fixed run through the selected or model-finest level."""
        ...

    def add_samples(
        self,
        additional_samples_per_level: Sequence[int],
    ) -> MLMCResult:
        """Add fixed sample counts to the active run."""
        ...

    @property
    def result(self) -> MLMCResult:
        """Return an immutable snapshot of the active run."""
        ...

    def run_to_tolerance(
        self,
        sampling_tolerance: float,
        *,
        pilot_samples_per_level: int,
        maximum_samples: int | None = None,
    ) -> MLMCResult:
        """Add correction samples until a sampling-error target is met."""
        ...